# LeetCode 1258 — Synonymous Sentences bằng Python

Link đề: https://leetcode.com/problems/synonymous-sentences/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Backtracking, Sort, Union-Find · 🔒 **Premium**

Skeleton LeetCode (theo lời giải Python trên doocs):

```python
class Solution:
    def generateSentences(self, synonyms: list[list[str]], text: str) -> list[str]:
        ...
```

> **Ghi chú Premium:** đề bài trong notebook được viết lại theo hiểu biết về bài (tham khảo https://leetcode.doocs.org/en/lc/1258/). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

1. Nhận ra quan hệ "đồng nghĩa" có **tính bắc cầu** ⇒ các từ chia thành **nhóm (thành phần liên thông)**.
2. Gom nhóm bằng ba cách: **DFS/BFS trên đồ thị từ**, **Union-Find**, hoặc tìm kiếm trực tiếp trên không gian câu (brute force).
3. Sinh mọi tổ hợp bằng **backtracking** (quay lui) và bằng `itertools.product` — hiểu đây là **tích Descartes** theo từng vị trí.
4. Hiểu vì sao sắp xếp từng nhóm từ theo từ điển là đủ để câu ra đúng thứ tự từ điển (và khi nào nên `sort` lại cho chắc).
5. Tránh lỗi: chỉ thay theo cặp trực tiếp (quên bắc cầu), quên `pop()` khi quay lui, quên giữ nguyên từ không có đồng nghĩa, quên sắp xếp.

## 2. Đề bài & ví dụ

Cho danh sách cặp từ tương đương `synonyms`, với `synonyms[i] = [s_i, t_i]` nghĩa là `s_i` và `t_i` là **đồng nghĩa**; và một câu `text` (các từ cách nhau đúng một dấu cách).

Trả về **mọi câu đồng nghĩa có thể** (thay mỗi từ bằng bất kỳ từ nào đồng nghĩa với nó, kể cả chính nó), **sắp xếp theo thứ tự từ điển**.

Quan hệ đồng nghĩa có tính **bắc cầu**: `happy ~ joy` và `joy ~ cheerful` ⇒ `happy ~ cheerful`.

**Ví dụ 1:** `synonyms = [["happy","joy"],["sad","sorrow"],["joy","cheerful"]]`, `text = "I am happy today but was sad yesterday"` → 6 câu (3 lựa chọn cho `happy` × 2 lựa chọn cho `sad`).

**Ví dụ 2:** `synonyms = [["happy","joy"],["cheerful","glad"]]`, cùng `text` → 2 câu (`happy`/`joy`; `sad` không có đồng nghĩa).

**Ràng buộc:**
- `0 <= synonyms.length <= 10`, mỗi từ dài 1..10, `s_i != t_i`, các cặp không trùng.
- `text` có tối đa 10 từ, cách nhau một dấu cách.

In [1]:
import random
import time
from collections import defaultdict, deque
from itertools import chain, product

TEXT = "I am happy today but was sad yesterday"
EX1 = ([["happy", "joy"], ["sad", "sorrow"], ["joy", "cheerful"]], TEXT,
       ["I am cheerful today but was sad yesterday", "I am cheerful today but was sorrow yesterday",
        "I am happy today but was sad yesterday", "I am happy today but was sorrow yesterday",
        "I am joy today but was sad yesterday", "I am joy today but was sorrow yesterday"])
EX2 = ([["happy", "joy"], ["cheerful", "glad"]], TEXT,
       ["I am happy today but was sad yesterday", "I am joy today but was sad yesterday"])
examples = [EX1, EX2]
for syn, text, want in examples:
    print("synonyms =", syn)
    for s in want:
        print("   ", s)

synonyms = [['happy', 'joy'], ['sad', 'sorrow'], ['joy', 'cheerful']]
    I am cheerful today but was sad yesterday
    I am cheerful today but was sorrow yesterday
    I am happy today but was sad yesterday
    I am happy today but was sorrow yesterday
    I am joy today but was sad yesterday
    I am joy today but was sorrow yesterday
synonyms = [['happy', 'joy'], ['cheerful', 'glad']]
    I am happy today but was sad yesterday
    I am joy today but was sad yesterday


## 3. Kiến thức nền

### 3.1 `str.split` và `' '.join`

`text.split()` tách câu thành danh sách từ; `' '.join(words)` ghép ngược lại. Với đề này (đúng một dấu cách) `split()` và `split(' ')` cho kết quả như nhau.

In [2]:
words = TEXT.split()
print(words)
words[2] = "joy"
print(" ".join(words))

['I', 'am', 'happy', 'today', 'but', 'was', 'sad', 'yesterday']
I am joy today but was sad yesterday


### 3.2 Đồ thị từ: danh sách kề bằng `defaultdict(set)`

Mỗi cặp `[a, b]` là một **cạnh vô hướng**. Nhóm đồng nghĩa của một từ = mọi từ đến được từ nó trong đồ thị.

In [3]:
adj = defaultdict(set)
for a, b in EX1[0]:
    adj[a].add(b)
    adj[b].add(a)
print(dict(adj))

def component(word, adj):
    seen = {word}
    stack = [word]
    while stack:
        w = stack.pop()
        for v in adj[w]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return sorted(seen)

print("Nhóm của 'happy':", component("happy", adj))
print("Nhóm của 'today':", component("today", adj), "(từ không có cặp -> chỉ chính nó)")

{'happy': {'joy'}, 'joy': {'cheerful', 'happy'}, 'sad': {'sorrow'}, 'sorrow': {'sad'}, 'cheerful': {'joy'}}
Nhóm của 'happy': ['cheerful', 'happy', 'joy']
Nhóm của 'today': ['today'] (từ không có cặp -> chỉ chính nó)


### 3.3 Union-Find trên chuỗi

Có thể chạy Union-Find trực tiếp với khoá là chuỗi bằng `dict` (thay cho mảng chỉ số). `find` trả về đại diện; `union` gộp hai nhóm.

In [4]:
uf = {}
def find(x):
    uf.setdefault(x, x)
    while uf[x] != x:
        uf[x] = uf[uf[x]]
        x = uf[x]
    return x
def union(a, b):
    uf[find(a)] = find(b)

for a, b in EX1[0]:
    union(a, b)
groups = defaultdict(list)
for w in uf:
    groups[find(w)].append(w)
print({k: sorted(v) for k, v in groups.items()})

{'cheerful': ['cheerful', 'happy', 'joy'], 'sorrow': ['sad', 'sorrow']}


### 3.4 Tích Descartes: `itertools.product`

`product(A, B, C)` sinh mọi bộ `(a, b, c)` với `a ∈ A, b ∈ B, c ∈ C`, theo thứ tự "đồng hồ đếm": phần tử cuối thay đổi nhanh nhất. Nếu mỗi danh sách đã sắp xếp, kết quả được sắp xếp theo thứ tự từ điển của **bộ**.

In [5]:
for t in product(["I"], ["cheerful", "happy", "joy"], ["sad", "sorrow"]):
    print(t)

('I', 'cheerful', 'sad')
('I', 'cheerful', 'sorrow')
('I', 'happy', 'sad')
('I', 'happy', 'sorrow')
('I', 'joy', 'sad')
('I', 'joy', 'sorrow')


### 3.5 Backtracking (quay lui) bằng một danh sách `path`

Khung quay lui: tại vị trí `i`, thử từng lựa chọn: `path.append(x)` → gọi đệ quy `i+1` → `path.pop()` để trả lại trạng thái. Đến `i == n` thì ghi nhận một kết quả.

In [6]:
def all_choices(options):
    res, path = [], []
    def bt(i):
        if i == len(options):
            res.append("".join(path))
            return
        for x in options[i]:
            path.append(x)
            bt(i + 1)
            path.pop()
    bt(0)
    return res

print(all_choices([["a", "b"], ["x"], ["1", "2"]]))

['ax1', 'ax2', 'bx1', 'bx2']


### 3.6 So sánh chuỗi trong Python

Python so sánh chuỗi theo **mã ký tự** (Unicode): chữ **hoa** `'A'..'Z'` (65–90) đứng **trước** chữ thường `'a'..'z'` (97–122), và dấu cách `' '` (32) đứng trước mọi chữ cái. Điều này quan trọng khi lý luận về thứ tự câu.

In [7]:
print(sorted(["joy", "Joy", "happy", "I", "glad"]))
print("'joy today' < 'joyful today':", "joy today" < "joyful today", "(vì ' ' < 'f')")

['I', 'Joy', 'glad', 'happy', 'joy']
'joy today' < 'joyful today': True (vì ' ' < 'f')


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**:

1. Gom nhóm: `happy—joy—cheerful` (bắc cầu qua `joy`) ⇒ nhóm `{cheerful, happy, joy}`; `sad—sorrow` ⇒ nhóm `{sad, sorrow}`.
2. Duyệt từng từ của câu:
   - `I`, `am`, `today`, `but`, `was`, `yesterday`: không thuộc nhóm nào ⇒ 1 lựa chọn.
   - `happy` ⇒ 3 lựa chọn (sắp xếp): `cheerful, happy, joy`.
   - `sad` ⇒ 2 lựa chọn: `sad, sorrow`.
3. Số câu = 3 × 2 = **6**. Liệt kê theo thứ tự: đổi vị trí sau (`sad`) nhanh hơn vị trí trước (`happy`).

Ví dụ 2: `cheerful—glad` là một nhóm riêng, không nối với `happy` ⇒ `happy` chỉ có `{happy, joy}` ⇒ 2 câu.

In [8]:
def options_per_word(synonyms, text):
    adj = defaultdict(set)
    for a, b in synonyms:
        adj[a].add(b); adj[b].add(a)
    return [component(w, adj) for w in text.split()]

for syn, text, want in examples:
    opts = options_per_word(syn, text)
    total = 1
    for o in opts:
        total *= len(o)
    print([o for o in opts if len(o) > 1], "-> số câu =", total, "| đề:", len(want))

[['cheerful', 'happy', 'joy'], ['sad', 'sorrow']] -> số câu = 6 | đề: 6
[['happy', 'joy']] -> số câu = 2 | đề: 2


## 5. Cách 1 — Brute Force / baseline: tìm kiếm trên không gian câu

**Ý tưởng:** coi mỗi **câu** là một đỉnh. Từ một câu, thay **một** từ bằng một đồng nghĩa **trực tiếp** (theo một cặp trong `synonyms`) ta được câu kề. BFS từ `text`, thu mọi câu đến được, rồi sắp xếp.

Không cần nghĩ tới tính bắc cầu: BFS tự đi qua nhiều bước thay thế (`happy → joy → cheerful`).

In [9]:
def generate_brute(synonyms, text):
    adj = defaultdict(set)
    for a, b in synonyms:
        adj[a].add(b); adj[b].add(a)
    seen = {text}
    q = deque([text])
    while q:
        s = q.popleft()
        words = s.split()
        for i, w in enumerate(words):
            for v in adj[w]:
                nxt = " ".join(words[:i] + [v] + words[i + 1:])
                if nxt not in seen:
                    seen.add(nxt)
                    q.append(nxt)
    return sorted(seen)

for syn, text, want in examples:
    got = generate_brute(syn, text)
    print(len(got), got == want)

6 True
2 True


**Độ phức tạp:** gọi `K` = số câu kết quả, `W` = số từ trong câu (≤ 10), `D` = bậc tối đa trong đồ thị từ, `L` = độ dài câu. Mỗi câu sinh `O(W·D)` câu kề, mỗi câu kề tốn `O(L)` để dựng/băm ⇒ **`O(K · W · D · L + K log K · L)`**. Bộ nhớ `O(K · L)`.

**Nút thắt:** cùng một câu bị "gặp lại" rất nhiều lần qua các đường thay thế khác nhau (thứ tự thay khác nhau cho cùng câu). Tốt hơn: tách bài toán thành (1) tìm nhóm của **từng từ**, (2) ghép các lựa chọn độc lập theo vị trí.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — độc lập theo vị trí.** Việc thay từ ở vị trí `i` không ảnh hưởng lựa chọn ở vị trí `j`. Nên tập câu = **tích Descartes** của các tập lựa chọn `options[0] × options[1] × … × options[W−1]`, với `options[i]` = nhóm đồng nghĩa của từ thứ `i` (hoặc `{từ đó}` nếu nó không có cặp nào).

**Insight 2 — nhóm = thành phần liên thông.** Đồng nghĩa có tính phản xạ, đối xứng, bắc cầu ⇒ là quan hệ tương đương ⇒ chia từ thành các lớp = thành phần liên thông của đồ thị cạnh `synonyms`.

**Insight 3 — thứ tự.** Nếu mỗi `options[i]` được sắp xếp tăng dần và ta sinh theo thứ tự "vị trí sau đổi nhanh hơn", thì danh sách câu ra theo thứ tự từ điển của **danh sách từ**. Vì dấu cách nhỏ hơn mọi chữ cái, thứ tự từ điển của danh sách từ trùng với thứ tự từ điển của câu đã ghép (với từ chỉ gồm chữ cái). Muốn chắc chắn tuyệt đối, cứ `sort` lại kết quả — chỉ tốn `O(K log K)`.

**Quy tắc "nếu… thì…":**
- Nếu từ không xuất hiện trong cặp nào ⇒ giữ nguyên (1 lựa chọn).
- Nếu từ thuộc một nhóm ⇒ thử **mọi** từ của nhóm, kể cả chính nó.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS tìm nhóm + `itertools.product`

1. Dựng đồ thị kề.
2. Với mỗi từ trong câu, lấy nhóm bằng DFS (có thể nhớ lại nhóm đã tính để không DFS lặp).
3. `product(*options)` rồi `' '.join`.

In [10]:
def generate_product(synonyms, text):
    adj = defaultdict(set)
    for a, b in synonyms:
        adj[a].add(b); adj[b].add(a)
    memo = {}
    def group(w):
        if w not in memo:
            comp = component(w, adj)          # đã sắp xếp
            for x in comp:
                memo[x] = comp
        return memo[w]
    options = [group(w) for w in text.split()]
    return [" ".join(t) for t in product(*options)]

for syn, text, want in examples:
    print(generate_product(syn, text) == want)

True
True


Bản nộp LeetCode của Cách 2 (thêm `sorted` ở cuối cho chắc chắn):

In [11]:
class SolutionProduct:
    def generateSentences(self, synonyms: list[list[str]], text: str) -> list[str]:
        adj = defaultdict(set)
        for a, b in synonyms:
            adj[a].add(b)
            adj[b].add(a)
        memo = {}
        def group(w):
            if w in memo:
                return memo[w]
            seen, stack = {w}, [w]
            while stack:
                x = stack.pop()
                for y in adj[x]:
                    if y not in seen:
                        seen.add(y)
                        stack.append(y)
            comp = sorted(seen)
            for x in comp:
                memo[x] = comp
            return comp
        options = [group(w) for w in text.split()]
        return sorted(" ".join(t) for t in product(*options))

print([SolutionProduct().generateSentences(s, t) == w for s, t, w in examples])

[True, True]


**Độ phức tạp Cách 2:** tìm nhóm `O(V + E)` (`V` từ trong cặp, `E = len(synonyms)`), sinh `K` câu mỗi câu `O(L)` ⇒ **`O(V + E + K·L)`** (+ `O(K log K · L)` nếu sort lại). Bộ nhớ `O(K·L)` cho kết quả.

### 7.2 Cách 3 — Union-Find + backtracking (lời giải doocs)

1. Đánh số mọi từ xuất hiện trong `synonyms`: `words`, `d[w] = chỉ số`.
2. Union-Find gộp từng cặp.
3. Nhóm `g[root]` = danh sách chỉ số, **sắp xếp theo từ** (`key=lambda i: words[i]`).
4. `dfs(i)`: nếu từ thứ `i` không có trong `d` ⇒ giữ nguyên; ngược lại thử mọi từ trong nhóm của nó. Hết câu ⇒ ghi nhận.

In [12]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
        self.size = [1] * n

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa != pb:
            if self.size[pa] > self.size[pb]:
                self.p[pb] = pa
                self.size[pa] += self.size[pb]
            else:
                self.p[pa] = pb
                self.size[pb] += self.size[pa]

def generate_uf(synonyms, text):
    words = list(set(chain.from_iterable(synonyms)))
    d = {w: i for i, w in enumerate(words)}
    uf = UnionFind(len(d))
    for a, b in synonyms:
        uf.union(d[a], d[b])
    g = defaultdict(list)
    for i in range(len(words)):
        g[uf.find(i)].append(i)
    for k in g:
        g[k].sort(key=lambda i: words[i])
    sentence = text.split()
    ans, t = [], []
    def dfs(i):
        if i >= len(sentence):
            ans.append(" ".join(t))
            return
        if sentence[i] not in d:
            t.append(sentence[i]); dfs(i + 1); t.pop()
        else:
            for j in g[uf.find(d[sentence[i]])]:
                t.append(words[j]); dfs(i + 1); t.pop()
    dfs(0)
    return ans

for syn, text, want in examples:
    print(generate_uf(syn, text) == want)

True
True


Bản nộp LeetCode (đúng tên lớp/hàm/tham số như doocs; lớp `UnionFind` đặt cùng file):

In [13]:
class SolutionUF:
    def generateSentences(self, synonyms: list[list[str]], text: str) -> list[str]:
        def dfs(i):
            if i >= len(sentence):
                ans.append(" ".join(t))
                return
            if sentence[i] not in d:
                t.append(sentence[i])
                dfs(i + 1)
                t.pop()
            else:
                root = uf.find(d[sentence[i]])
                for j in g[root]:
                    t.append(words[j])
                    dfs(i + 1)
                    t.pop()

        words = list(set(chain.from_iterable(synonyms)))
        d = {w: i for i, w in enumerate(words)}
        uf = UnionFind(len(d))
        for a, b in synonyms:
            uf.union(d[a], d[b])
        g = defaultdict(list)
        for i in range(len(words)):
            g[uf.find(i)].append(i)
        for k in g.keys():
            g[k].sort(key=lambda i: words[i])
        sentence = text.split()
        ans = []
        t = []
        dfs(0)
        return ans

print([SolutionUF().generateSentences(s, t) == w for s, t, w in examples])

[True, True]


**Độ phức tạp Cách 3:** Union-Find `O(E·α(V))`, sắp nhóm `O(V log V)`, quay lui sinh `K` câu, mỗi lá tốn `O(L)` để `join` ⇒ **`O(V log V + K·L)`**. Bộ nhớ: `O(V)` + độ sâu đệ quy `O(W)` + kết quả `O(K·L)`.

### 7.3 Cách 4 — Mở rộng dần theo vị trí (BFS theo tầng)

Không đệ quy: bắt đầu `partial = [[]]`; với mỗi vị trí `i`, thay `partial` bằng `[p + [x] for p in partial for x in options[i]]`. Thứ tự vòng lặp (`p` ngoài, `x` trong) giữ đúng thứ tự từ điển.

In [14]:
def generate_iterative(synonyms, text):
    uf = {}
    def find(x):
        uf.setdefault(x, x)
        while uf[x] != x:
            uf[x] = uf[uf[x]]
            x = uf[x]
        return x
    for a, b in synonyms:
        uf[find(a)] = find(b)
    groups = defaultdict(list)
    for w in list(uf):
        groups[find(w)].append(w)
    for k in groups:
        groups[k].sort()
    partial = [[]]
    for w in text.split():
        opts = groups[find(w)] if w in uf else [w]
        partial = [p + [x] for p in partial for x in opts]
    return [" ".join(p) for p in partial]

for syn, text, want in examples:
    print(generate_iterative(syn, text) == want)

True
True


**Độ phức tạp Cách 4:** tổng kích thước các danh sách trung gian ≤ `K·W` từ ⇒ `O(K·W + K·L)` ≈ `O(K·L)`; bộ nhớ `O(K·W)`.

**Cận trên của `K`:** tối đa 10 cặp ⇒ tối đa 20 từ phân biệt. Câu có ≤ 10 từ và **một từ có thể lặp lại**. Ví dụ xấu: 10 cặp nối thành một dây chuyền 11 từ (một nhóm duy nhất cỡ 11); nếu cả 10 từ trong câu đều thuộc nhóm này thì `K = 11^10` (≈ 2.6·10¹⁰)! Trên thực tế bộ test của LeetCode nhỏ, nhưng cần biết **đầu ra có thể lớn theo cấp số mũ** — mọi thuật toán đều phải tốn ít nhất `Ω(K·L)`.

## 8. TRACE từng bước

Trace **Cách 3 (backtracking)** trên Ví dụ 1. Cây quyết định chỉ phân nhánh ở `happy` (vị trí 2) và `sad` (vị trí 6):

| Vị trí `i` | từ gốc | lựa chọn | `t` sau khi chọn |
|---|---|---|---|
| 0–1 | I, am | giữ | `[I, am]` |
| 2 | happy | cheerful / happy / joy | `[I, am, cheerful]` … |
| 3–5 | today, but, was | giữ | |
| 6 | sad | sad / sorrow | |
| 7 | yesterday | giữ | ⇒ ghi nhận câu |

In [15]:
def trace_backtrack(synonyms, text):
    uf = {}
    def find(x):
        uf.setdefault(x, x)
        while uf[x] != x:
            x = uf[x]
        return x
    for a, b in synonyms:
        uf[find(a)] = find(b)
    groups = defaultdict(list)
    for w in list(uf):
        groups[find(w)].append(w)
    for k in groups:
        groups[k].sort()
    sentence = text.split()
    ans, t = [], []
    def dfs(i, depth):
        if i == len(sentence):
            ans.append(" ".join(t))
            print("  " * depth + f"✔ ghi nhận #{len(ans)}: {ans[-1]!r}")
            return
        opts = groups[find(sentence[i])] if sentence[i] in uf else [sentence[i]]
        if len(opts) > 1:
            print("  " * depth + f"i={i} {sentence[i]!r}: phân nhánh {opts}")
        for x in opts:
            t.append(x)
            if len(opts) > 1:
                print("  " * depth + f"  chọn {x!r}, t = {t}")
            dfs(i + 1, depth + (len(opts) > 1))
            t.pop()
    dfs(0, 0)
    return ans

res = trace_backtrack(EX1[0], "I am happy but sad")

i=2 'happy': phân nhánh ['cheerful', 'happy', 'joy']
  chọn 'cheerful', t = ['I', 'am', 'cheerful']
  i=4 'sad': phân nhánh ['sad', 'sorrow']
    chọn 'sad', t = ['I', 'am', 'cheerful', 'but', 'sad']
    ✔ ghi nhận #1: 'I am cheerful but sad'
    chọn 'sorrow', t = ['I', 'am', 'cheerful', 'but', 'sorrow']
    ✔ ghi nhận #2: 'I am cheerful but sorrow'
  chọn 'happy', t = ['I', 'am', 'happy']
  i=4 'sad': phân nhánh ['sad', 'sorrow']
    chọn 'sad', t = ['I', 'am', 'happy', 'but', 'sad']
    ✔ ghi nhận #3: 'I am happy but sad'
    chọn 'sorrow', t = ['I', 'am', 'happy', 'but', 'sorrow']
    ✔ ghi nhận #4: 'I am happy but sorrow'
  chọn 'joy', t = ['I', 'am', 'joy']
  i=4 'sad': phân nhánh ['sad', 'sorrow']
    chọn 'sad', t = ['I', 'am', 'joy', 'but', 'sad']
    ✔ ghi nhận #5: 'I am joy but sad'
    chọn 'sorrow', t = ['I', 'am', 'joy', 'but', 'sorrow']
    ✔ ghi nhận #6: 'I am joy but sorrow'


In [16]:
def trace_iterative(synonyms, text):
    opts_all = options_per_word(synonyms, text)
    partial = [[]]
    for i, opts in enumerate(opts_all):
        partial = [p + [x] for p in partial for x in opts]
        print(f"sau vị trí {i} ({opts}): {len(partial)} câu dở dang, ví dụ {[' '.join(p) for p in partial[:3]]}")
    return [" ".join(p) for p in partial]

trace_iterative(EX1[0], "I am happy but sad");

sau vị trí 0 (['I']): 1 câu dở dang, ví dụ ['I']
sau vị trí 1 (['am']): 1 câu dở dang, ví dụ ['I am']
sau vị trí 2 (['cheerful', 'happy', 'joy']): 3 câu dở dang, ví dụ ['I am cheerful', 'I am happy', 'I am joy']
sau vị trí 3 (['but']): 3 câu dở dang, ví dụ ['I am cheerful but', 'I am happy but', 'I am joy but']
sau vị trí 4 (['sad', 'sorrow']): 6 câu dở dang, ví dụ ['I am cheerful but sad', 'I am cheerful but sorrow', 'I am happy but sad']


## 9. Code nộp LeetCode

Dùng Cách 3 (Union-Find + backtracking) đúng như doocs; lớp `UnionFind` khai báo trước lớp `Solution`:

In [17]:
from collections import defaultdict
from itertools import chain

class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
        self.size = [1] * n

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa != pb:
            if self.size[pa] > self.size[pb]:
                self.p[pb] = pa
                self.size[pa] += self.size[pb]
            else:
                self.p[pa] = pb
                self.size[pb] += self.size[pa]


class Solution:
    def generateSentences(self, synonyms: list[list[str]], text: str) -> list[str]:
        def dfs(i):
            if i >= len(sentence):
                ans.append(" ".join(t))
                return
            if sentence[i] not in d:            # không có đồng nghĩa: giữ nguyên
                t.append(sentence[i])
                dfs(i + 1)
                t.pop()
            else:                               # thử mọi từ trong nhóm (đã sắp xếp)
                root = uf.find(d[sentence[i]])
                for j in g[root]:
                    t.append(words[j])
                    dfs(i + 1)
                    t.pop()

        words = list(set(chain.from_iterable(synonyms)))
        d = {w: i for i, w in enumerate(words)}
        uf = UnionFind(len(d))
        for a, b in synonyms:
            uf.union(d[a], d[b])
        g = defaultdict(list)
        for i in range(len(words)):
            g[uf.find(i)].append(i)
        for k in g.keys():
            g[k].sort(key=lambda i: words[i])
        sentence = text.split()
        ans = []
        t = []
        dfs(0)
        return ans

for syn, text, want in examples:
    print(Solution().generateSentences(syn, text) == want)

True
True


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Hai từ `u, v` đồng nghĩa (theo bao đóng phản xạ–đối xứng–bắc cầu của `synonyms`) ⇔ `u, v` cùng thành phần liên thông của đồ thị có cạnh là các cặp. Union-Find / DFS tính đúng các thành phần này.

**Bổ đề 2.** Một câu `s'` là "câu đồng nghĩa" của `text` ⇔ `s'` có cùng số từ và với mọi `i`, từ `s'[i]` đồng nghĩa với `text[i]`. (Mỗi phép thay một từ chỉ đổi một vị trí trong cùng lớp; ngược lại có thể đổi từng vị trí một.) Đây cũng là lý do Cách 1 (BFS trên câu) và các cách còn lại cho cùng một tập.

**Đầy đủ & không trùng lặp.** Quay lui duyệt đúng một lần mỗi bộ `(x_0, …, x_{W−1})` với `x_i ∈ options[i]`; các `options[i]` là tập (không trùng) ⇒ mỗi câu sinh đúng một lần.

**Thứ tự.** Với các `options[i]` tăng dần, quay lui sinh các bộ theo thứ tự từ điển trên **bộ từ** (vị trí đầu tiên khác nhau quyết định). Hai câu có bộ từ khác nhau lần đầu ở vị trí `i` có cùng tiền tố ký tự trước đó; tại vị trí `i`, nếu `x < y` và `x` là tiền tố của `y` thì ký tự tiếp theo của câu chứa `x` là dấu cách (hoặc hết câu) — nhỏ hơn mọi chữ cái của `y` ⇒ thứ tự câu trùng thứ tự bộ. ∎

## 11. Phân tích Time / Space

Ký hiệu: `E = len(synonyms) ≤ 10`, `V ≤ 2E` từ trong cặp, `W ≤ 10` từ trong câu, `L` = độ dài câu, `K` = số câu kết quả.

| Cách | Time | Space (ngoài đầu ra) |
|---|---|---|
| 1. BFS trên câu | `O(K·W·D·L + K log K·L)` | `O(K·L)` (tập `seen`) |
| 2. DFS nhóm + product | `O(V + E + K·L)` (+ sort) | `O(V)` |
| 3. Union-Find + backtracking | `O(V log V + K·L)` | `O(V + W)` |
| 4. Mở rộng theo tầng | `O(K·L)` | `O(K·W)` trung gian |

Vì đầu ra có `K` câu dài `L`, mọi thuật toán cần **ít nhất `Ω(K·L)`** — Cách 2/3/4 đạt cận này. `K` có thể lớn theo cấp số mũ của `W` (worst case), còn average trên test thực tế rất nhỏ.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {
    "brute": generate_brute,
    "product": generate_product,
    "product_submit": lambda s, t: SolutionProduct().generateSentences(s, t),
    "uf": generate_uf,
    "iterative": generate_iterative,
    "submit": lambda s, t: Solution().generateSentences(s, t),
}
edge = [
    ([], "hello world", ["hello world"]),                              # không có cặp
    ([["a", "b"]], "a", ["a", "b"]),                                   # 1 từ
    ([["a", "b"]], "c", ["c"]),                                        # từ không liên quan
    ([["a", "b"]], "a a", ["a a", "a b", "b a", "b b"]),               # từ lặp lại
    ([["a", "b"], ["b", "c"], ["c", "d"]], "d", ["a", "b", "c", "d"]), # dây chuyền bắc cầu
    ([["joy", "joyful"]], "joy x", ["joy x", "joyful x"]),             # từ là tiền tố của từ khác
    ([["b", "A"]], "b", ["A", "b"]),                                   # chữ hoa đứng trước chữ thường
]
for name, f in ALL.items():
    for syn, text, want in examples + edge:
        got = f(syn, text)
        assert got == want, (name, syn, text, got, want)
print("Mọi cách đúng với", len(examples + edge), "test (so khớp chính xác cả thứ tự)")

Mọi cách đúng với 9 test (so khớp chính xác cả thứ tự)


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh từ vựng nhỏ (để có nhiều bắc cầu), ≤ 10 cặp không trùng, câu ≤ 6 từ (giữ `K` vừa phải). Kiểm tra: kết quả khớp brute force, **đã sắp xếp**, **không trùng lặp**.

In [19]:
def random_case(rng):
    vocab = [rng.choice("abcdefgh") * rng.randint(1, 2) + rng.choice(["", "x", "y"]) for _ in range(12)]
    vocab = sorted(set(vocab))
    pairs = set()
    for _ in range(rng.randint(0, 10)):
        a, b = rng.sample(vocab, 2)
        if (a, b) not in pairs and (b, a) not in pairs:
            pairs.add((a, b))
    synonyms = [list(p) for p in pairs]
    text = " ".join(rng.choice(vocab) for _ in range(rng.randint(1, 6)))
    return synonyms, text

rng = random.Random(1258)
for _ in range(300):
    syn, text = random_case(rng)
    want = generate_brute(syn, text)
    assert want == sorted(set(want))
    for name, f in ALL.items():
        assert f(syn, text) == want, (name, syn, text)
print("300 test ngẫu nhiên: mọi cách khớp brute force, kết quả đã sort và không trùng")

300 test ngẫu nhiên: mọi cách khớp brute force, kết quả đã sort và không trùng


### 12.3 Trường hợp "nặng": nhiều từ cùng nhóm lớn

In [20]:
syn = [[f"w{i}", f"w{i+1}"] for i in range(10)]          # 11 từ cùng một nhóm
text = "w0 is w5 and w10"                                  # 3 vị trí thuộc nhóm -> 11^3 câu
for name in ["product_submit", "uf", "iterative", "submit", "brute"]:
    t0 = time.perf_counter()
    res = ALL[name](syn, text)
    print(f"{name:15}: {len(res)} câu, {time.perf_counter() - t0:.3f}s")
assert len(res) == 11 ** 3

product_submit : 1331 câu, 0.000s
uf             : 1331 câu, 0.000s
iterative      : 1331 câu, 0.000s
submit         : 1331 câu, 0.000s
brute          : 1331 câu, 0.003s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ thay theo cặp trực tiếp (quên tính bắc cầu)

Dựng `dict` từ → các đồng nghĩa **trực tiếp** rồi chỉ thay một bước. Ví dụ 1 sẽ thiếu `cheerful` (chỉ nối với `joy`, không nối trực tiếp với `happy`).

In [21]:
def generate_wrong1(synonyms, text):
    adj = defaultdict(set)
    for a, b in synonyms:
        adj[a].add(b); adj[b].add(a)
    options = [sorted({w} | adj[w]) for w in text.split()]     # SAI: chỉ 1 bước
    return [" ".join(t) for t in product(*options)]

got = generate_wrong1(*EX1[:2])
print("Sai :", len(got), "câu ->", [s for s in got if "sad" in s])
print("Đúng:", len(EX1[2]), "câu; bị thiếu:", sorted(set(EX1[2]) - set(got)))

Sai : 4 câu -> ['I am happy today but was sad yesterday', 'I am joy today but was sad yesterday']
Đúng: 6 câu; bị thiếu: ['I am cheerful today but was sad yesterday', 'I am cheerful today but was sorrow yesterday']


### Lỗi 2 — Quên `t.pop()` trong quay lui

Không trả lại trạng thái ⇒ danh sách `t` dài ra mãi, các câu sau dính từ của câu trước.

In [22]:
def generate_wrong2(synonyms, text):
    opts = options_per_word(synonyms, text)
    ans, t = [], []
    def dfs(i):
        if i == len(opts):
            ans.append(" ".join(t))
            return
        for x in opts[i]:
            t.append(x)
            dfs(i + 1)                 # SAI: thiếu t.pop()
    dfs(0)
    return ans

for s in generate_wrong2([["a", "b"]], "a c")[:3]:
    print("Sai :", repr(s))
print("Đúng:", Solution().generateSentences([["a", "b"]], "a c"))

Sai : 'a c'
Sai : 'a c b c'
Đúng: ['a c', 'b c']


### Lỗi 3 — Quên sắp xếp (hoặc sắp nhóm theo chỉ số thay vì theo chữ)

Dùng `set` để lưu nhóm rồi duyệt trực tiếp ⇒ thứ tự tuỳ thuộc băm; hoặc trong lời giải Union-Find, `g[k].sort()` sắp theo **chỉ số** thay vì `key=lambda i: words[i]`. Để kết quả tái lập được trong demo, ta mô phỏng bằng việc sắp nhóm theo **thứ tự xuất hiện trong `synonyms`**.

In [23]:
def generate_wrong3(synonyms, text):
    order = list(dict.fromkeys(chain.from_iterable(synonyms)))   # thứ tự xuất hiện
    pos = {w: i for i, w in enumerate(order)}
    adj = defaultdict(set)
    for a, b in synonyms:
        adj[a].add(b); adj[b].add(a)
    options = [sorted(component(w, adj), key=lambda x: pos.get(x, 0)) for w in text.split()]  # SAI
    return [" ".join(t) for t in product(*options)]

got = generate_wrong3(*EX1[:2])
print("Sai (3 câu đầu):", got[:3])
print("Đúng (3 câu đầu):", EX1[2][:3])
print("Cùng tập câu nhưng sai thứ tự?", sorted(got) == EX1[2] and got != EX1[2])

Sai (3 câu đầu): ['I am happy today but was sad yesterday', 'I am happy today but was sorrow yesterday', 'I am joy today but was sad yesterday']
Đúng (3 câu đầu): ['I am cheerful today but was sad yesterday', 'I am cheerful today but was sorrow yesterday', 'I am happy today but was sad yesterday']
Cùng tập câu nhưng sai thứ tự? True


### Lỗi 4 — Bỏ mất từ không có đồng nghĩa

Chỉ ghép những từ có trong `d`/`groups` (ví dụ dùng `groups.get(w, [])`) ⇒ một từ không có cặp cho danh sách lựa chọn **rỗng** ⇒ tích Descartes rỗng ⇒ trả về `[]`.

In [24]:
def generate_wrong4(synonyms, text):
    adj = defaultdict(set)
    for a, b in synonyms:
        adj[a].add(b); adj[b].add(a)
    groups = {w: component(w, adj) for w in list(adj)}
    options = [groups.get(w, []) for w in text.split()]      # SAI: [] cho từ lẻ
    return [" ".join(t) for t in product(*options)]

print("Sai :", generate_wrong4(*EX2[:2]))
print("Đúng:", Solution().generateSentences(*EX2[:2]))

Sai : []
Đúng: ['I am happy today but was sad yesterday', 'I am joy today but was sad yesterday']


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"equivalent", "synonym", "bắc cầu", "cùng nhóm"** ⇒ Union-Find hoặc DFS thành phần liên thông (bài 721 Accounts Merge, 1061 Lexicographically Smallest Equivalent String).
- **"Trả về mọi tổ hợp / mọi câu có thể"** ⇒ backtracking hoặc `itertools.product`; độ phức tạp chi phối bởi kích thước đầu ra.
- Lựa chọn ở các vị trí **độc lập** ⇒ tích Descartes, không cần BFS trên toàn bộ trạng thái.
- Kết quả cần "sorted lexicographically" ⇒ sắp từng nhóm trước khi sinh; nếu lười lập luận, `sorted(...)` ở cuối luôn an toàn.
- Python: `chain.from_iterable(pairs)` làm phẳng danh sách cặp; `dict.setdefault` hữu ích cho Union-Find trên chuỗi.

## 15. Biến thể của bài

1. **Chỉ đếm số câu** (không liệt kê): tích kích thước nhóm ⇒ `O(V + E + W)`, có thể rất lớn ⇒ có thể lấy modulo.
2. **Câu nhỏ nhất theo từ điển**: thay mỗi từ bằng từ nhỏ nhất trong nhóm (giống bài 1061).
3. **Câu thứ `k`** (0-index) trong danh sách đã sắp: biểu diễn `k` theo **hệ cơ số hỗn hợp** với cơ số là kích thước các nhóm.
4. **Đồng nghĩa có hướng** (chỉ `s → t`, không ngược lại) ⇒ thay thành phần liên thông bằng **tập đến được** trên đồ thị có hướng.
5. **LeetCode 721 — Accounts Merge**: cùng kỹ thuật gom nhóm chuỗi bằng Union-Find.

Minh hoạ biến thể 1 & 3:

In [25]:
def count_sentences(synonyms, text):
    total = 1
    for opts in options_per_word(synonyms, text):
        total *= len(opts)
    return total

def kth_sentence(synonyms, text, k):
    opts = options_per_word(synonyms, text)
    res = []
    for o in reversed(opts):          # vị trí cuối thay đổi nhanh nhất
        k, r = divmod(k, len(o))
        res.append(o[r])
    return " ".join(reversed(res))

print(count_sentences(*EX1[:2]))
print([kth_sentence(*EX1[:2], k) for k in range(6)] == EX1[2])

6
True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trên câu | Thay 1 từ bằng đồng nghĩa trực tiếp, BFS mọi câu, sort | `O(K·W·D·L)` + sort | `O(K·L)` | Kiểm chứng, không cần nghĩ về nhóm |
| 2. DFS nhóm + `product` | Tìm nhóm từng từ, tích Descartes | `O(V+E+K·L)` | `O(V)` | Ngắn gọn, "Pythonic" |
| 3. Union-Find + backtracking | Gom nhóm bằng DSU, quay lui theo vị trí | `O(V log V + K·L)` | `O(V+W)` | Lời giải chuẩn, dễ mở rộng (cắt tỉa, điều kiện thêm) |
| 4. Mở rộng theo tầng | `partial = [p+[x] ...]` từng vị trí | `O(K·L)` | `O(K·W)` | Khi tránh đệ quy |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().generateSentences([["a","b"],["c","d"]], "a c")` trả về gì (đúng thứ tự)?

### Bài 2 (tính tay)
Với `synonyms = [["big","large"],["large","huge"],["small","tiny"]]` và `text = "a big small big"`, có bao nhiêu câu? Câu đầu tiên và câu cuối cùng là gì?

### Bài 3 (điền TODO)
Hoàn thiện `smallest_sentence(synonyms, text)`: trả về câu **nhỏ nhất** theo từ điển (không liệt kê hết).

### Bài 4 (tự viết lại)
Viết lại lời giải bằng **DFS gom nhóm + backtracking** (không dùng `product`, không dùng Union-Find): `generate_dfs_bt(synonyms, text)`.

### Bài 5 (mini challenge)
Viết `generate_limited(synonyms, text, k)`: trả về **k câu đầu tiên** theo thứ tự từ điển mà **không** sinh toàn bộ (dừng quay lui sớm).

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def smallest_sentence(synonyms, text):
    # TODO: với mỗi từ, chọn từ nhỏ nhất trong nhóm của nó
    pass

# Bài 4 — TODO
def generate_dfs_bt(synonyms, text):
    # TODO 1: dựng đồ thị kề, hàm group(w) trả về nhóm đã sắp xếp
    # TODO 2: quay lui theo vị trí, nhớ pop()
    pass

# Bài 5 — TODO
def generate_limited(synonyms, text, k):
    # TODO: quay lui như Cách 3, dừng khi len(ans) == k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Nhóm `{a, b}` và `{c, d}`. (2) Vị trí sau đổi nhanh hơn. (3) `["a c", "a d", "b c", "b d"]`.

**Bài 2.** (1) Nhóm `{big, huge, large}` (bắc cầu), `{small, tiny}`. (2) `big` xuất hiện 2 lần, mỗi lần 3 lựa chọn: `1 × 3 × 2 × 3 = 18`. (3) Đầu: `"a big small big"`; cuối: `"a large tiny large"`.

**Bài 3.** (1) Câu nhỏ nhất = chọn từ nhỏ nhất ở **từng** vị trí (vì vị trí trước quyết định trước). (2) `min(group(w))`. (3) Không cần sinh tổ hợp: `O(V + E + W)`.

**Bài 4.** (1) Tái dùng `component` ở mục 3.2 + `memo`. (2) Hàm `bt(i)` như mục 3.5. (3) So sánh với `Solution` trên test ngẫu nhiên.

**Bài 5.** (1) Cho `dfs` trả `True` khi đã đủ `k` câu. (2) Sau mỗi lời gọi con, nếu `True` thì `return True` ngay (nhưng vẫn `pop` trước khi thoát — hoặc chấp nhận `t` bẩn vì không dùng nữa). (3) Kết quả phải bằng `Solution().generateSentences(...)[:k]`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().generateSentences([["a", "b"], ["c", "d"]], "a c") == ["a c", "a d", "b c", "b d"]
# Bài 2
B2 = [["big", "large"], ["large", "huge"], ["small", "tiny"]]
r = Solution().generateSentences(B2, "a big small big")
assert len(r) == 18 and r[0] == "a big small big" and r[-1] == "a large tiny large"

# Bài 3
def smallest_sentence_sol(synonyms, text):
    return " ".join(opts[0] for opts in options_per_word(synonyms, text))

# Bài 4
def generate_dfs_bt_sol(synonyms, text):
    adj = defaultdict(set)
    for a, b in synonyms:
        adj[a].add(b); adj[b].add(a)
    memo = {}
    def group(w):
        if w not in memo:
            comp = component(w, adj)
            for x in comp:
                memo[x] = comp
        return memo[w]
    opts = [group(w) for w in text.split()]
    ans, t = [], []
    def bt(i):
        if i == len(opts):
            ans.append(" ".join(t)); return
        for x in opts[i]:
            t.append(x); bt(i + 1); t.pop()
    bt(0)
    return ans

# Bài 5
def generate_limited_sol(synonyms, text, k):
    opts = options_per_word(synonyms, text)
    ans, t = [], []
    def dfs(i):
        if i == len(opts):
            ans.append(" ".join(t))
            return len(ans) >= k
        for x in opts[i]:
            t.append(x)
            stop = dfs(i + 1)
            t.pop()
            if stop:
                return True
        return False
    if k > 0:
        dfs(0)
    return ans

rng = random.Random(77)
for _ in range(200):
    syn, text = random_case(rng)
    full = Solution().generateSentences(syn, text)
    assert smallest_sentence_sol(syn, text) == full[0]
    assert generate_dfs_bt_sol(syn, text) == full
    k = rng.randint(0, len(full) + 2)
    assert generate_limited_sol(syn, text, k) == full[:k]
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "gom nhóm tương đương + sinh tổ hợp theo vị trí":**

```python
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b in pairs:
    parent[find(a)] = find(b)

groups = defaultdict(list)
for w in list(parent):
    groups[find(w)].append(w)
for g in groups.values():
    g.sort()

options = [groups[find(w)] if w in parent else [w] for w in text.split()]
ans = [' '.join(t) for t in product(*options)]
```

**Checklist trước khi submit:**
- [ ] Tính **bắc cầu**: dùng thành phần liên thông, không chỉ cặp trực tiếp.
- [ ] Từ không có cặp nào vẫn được giữ (1 lựa chọn, không phải 0).
- [ ] Mỗi nhóm **sắp xếp theo chữ** (không theo chỉ số / thứ tự băm).
- [ ] Quay lui có `append` → đệ quy → `pop`.
- [ ] Test: `synonyms = []`, từ lặp lại trong câu, từ là tiền tố của từ khác.
- [ ] Đúng chữ ký: `generateSentences(self, synonyms, text) -> list[str]`.